#  Learning Style: Myth or Reality?
## A Multi-Task Predictive Audit of Student Performance

In [7]:
import pandas as pd
import missingno as msno

### 2. Research Questions
Your project must provide an evidence-based answer to each of the following questions. The
answers must emerge from your experimental results, not from prior assumptions or literature
alone.
1. RQ1. Can a machine learning model predict a student's Exam Score and Final Grade
from their behavioural and motivational profile with meaningful accuracy, and which
model family performs best?
2. RQ2. Does the LearningStyle label add statistically meaningful predictive power once all
other features are available, as measured by an ablation experiment?
3. RQ3. Are the LearningStyle labels in the dataset independently derived, or do they
largely recapitulate structure already visible in the behavioural features — and how do
you know?
4. RQ4. If a school used your best model to intervene on at-risk students, what are the
realistic harms, who bears them, and how should the model be constrained?


### STEP 1: Data Loading, Audit, and Preprocessing

In [ ]:
line = 60 * '-'
df = pd.read_csv('student_performance.csv')

n_rows, n_cols = df.shape

print(f"O dataset tem {n_rows} linhas e {n_cols} colunas.")
print('\n' + line + '\n')

print(df.info())

df.describe()

O dataset tem 14003 linhas e 16 colunas.

------------------------------------------------------------

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14003 entries, 0 to 14002
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   StudyHours            14003 non-null  int64
 1   Attendance            14003 non-null  int64
 2   Resources             14003 non-null  int64
 3   Extracurricular       14003 non-null  int64
 4   Motivation            14003 non-null  int64
 5   Internet              14003 non-null  int64
 6   Gender                14003 non-null  int64
 7   Age                   14003 non-null  int64
 8   LearningStyle         14003 non-null  int64
 9   OnlineCourses         14003 non-null  int64
 10  Discussions           14003 non-null  int64
 11  AssignmentCompletion  14003 non-null  int64
 12  ExamScore             14003 non-null  int64
 13  EduTech               14003 non-null  int64
 14

,StudyHours,Attendance,Resources,Extracurricular,Motivation,Internet,Gender,Age,LearningStyle,OnlineCourses,Discussions,AssignmentCompletion,ExamScore,EduTech,StressLevel,FinalGrade
count,14003.000000,14003.000000,14003.000000,14003.000000,14003.000000,14003.000000,14003.000000,14003.000000,14003.000000,14003.000000,14003.00000,14003.000000,14003.000000,14003.000000,14003.000000,14003.000000
mean,19.987431,80.194316,1.104406,0.594158,0.905806,0.925516,0.551953,23.532172,1.515461,9.891952,0.60587,74.502535,70.346926,0.709062,1.304363,1.447904
std,5.890637,11.472181,0.697362,0.491072,0.695896,0.262566,0.497311,3.514293,1.112941,6.112801,0.48868,14.632177,17.688113,0.454211,0.785383,1.121550
min,5.000000,60.000000,0.000000,0.000000,0.000000,0.000000,0.000000,18.000000,0.000000,0.000000,0.00000,50.000000,40.000000,0.000000,0.000000,0.000000
25%,16.000000,70.000000,1.000000,0.000000,0.000000,1.000000,0.000000,20.000000,1.000000,5.000000,0.00000,62.000000,55.000000,0.000000,1.000000,0.000000
50%,20.000000,80.000000,1.000000,1.000000,1.000000,1.000000,1.000000,24.000000,2.000000,10.000000,1.00000,74.000000,70.000000,1.000000,2.000000,1.000000
75%,24.000000,90.000000,2.000000,1.000000,1.000000,1.000000,1.000000,27.000000,3.000000,15.000000,1.00000,87.000000,86.000000,1.000000,2.000000,2.000000
max,44.000000,100.000000,2.000000,1.000000,2.000000,1.000000,1.000000,29.000000,3.000000,20.000000,1.00000,100.000000,100.000000,1.000000,2.000000,3.000000


In [6]:
df.head()

,StudyHours,Attendance,Resources,Extracurricular,Motivation,Internet,Gender,Age,LearningStyle,OnlineCourses,Discussions,AssignmentCompletion,ExamScore,EduTech,StressLevel,FinalGrade
0,19,64,1,0,0,1,0,19,2,8,1,59,40,0,1,3
1,19,64,1,0,0,1,0,23,3,16,0,90,66,0,1,2
2,19,64,1,0,0,1,0,28,1,19,0,67,99,1,1,0
3,19,64,1,1,0,1,0,19,2,8,1,59,40,0,1,3
4,19,64,1,1,0,1,0,23,3,16,0,90,66,0,1,2


In [11]:
missing = df.isnull().sum()
pct = (missing / len(df) * 100).round(1)
summary = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': pct
})

if summary['missing_count'].sum() == 0:
    print("Não há valores ausentes no dataset.")
else:
    print("Valores ausentes por coluna:")
    print(summary[summary['missing_count'] > 0])

Não há valores ausentes no dataset.


In [12]:
duplicates = df.duplicated().sum()

print(f"Número de linhas duplicadas: {duplicates}")

Número de linhas duplicadas: 1534


### Key Features

- **Study behaviors & engagement →** StudyHours, Attendance, Extracurricular, AssignmentCompletion, OnlineCourses, Discussions
- **Resources & environment →** Resources, Internet, EduTech
- **Motivation & psychology →** Motivation, StressLevel
- **Demographics →** Gender, Age (18–30 years)
- **Learning preference →** LearningStyle
- **Performance indicators →** ExamScore, FinalGrade

### Column Descriptions

- **StudyHours** – Number of study hours per week.
- **Attendance** – Percentage of classes attended.
- **Resources** – Availability and use of academic resources (e.g., library, notes).
- **Extracurricular** – Participation in extracurricular activities (Yes/No).
- **Motivation** – Self-reported motivation level (numeric scale).
- **Internet** – Access to the internet for study purposes (Yes/No).
- **Gender** – Student’s gender (Male/Female).
- **Age** – Age of student (18–30 years).
- **LearningStyle** – Preferred learning style (e.g., Visual, Auditory, Kinesthetic, Reading/Writing).
- **OnlineCourses** – Participation in online courses (Yes/No).
- **Discussions** – Engagement in study group discussions or forums.
- **AssignmentCompletion** – Rate of completing assignments on time (numeric scale).
- **ExamScore** – Score obtained in the main exam.
- **EduTech** – Usage of educational technology tools/platforms.
- **StressLevel** – Self-reported stress level (numeric scale).
- **FinalGrade** – Final course grade (target variable for prediction).